### Generate predictions on Colab

In [ ]:
# @title Run parameters
import os

# Browser or VS Code Colab: uncomment these lines and set the values.
# Colab CLI: keep them commented and pass the values with `colab exec --env KEY=VALUE`.
# %env MODEL=Qwen/Qwen2.5-Coder-1.5B-Instruct
# %env SPLIT=dev
# %env RUN=baseline
# %env BATCH_SIZE=16

names = ("MODEL", "SPLIT", "RUN", "BATCH_SIZE")
missing = [name for name in names if name not in os.environ]
if missing:
    raise RuntimeError(
        f"Set {', '.join(missing)}: colab exec --env KEY=VALUE, or %env KEY=VALUE in a cell above"
    )

MODEL, SPLIT, RUN = os.environ["MODEL"], os.environ["SPLIT"], os.environ["RUN"]
BATCH_SIZE = int(os.environ["BATCH_SIZE"])

REPO = "unbiasederudite/qwen2.5-coder-text2sql-qlora"  # GitHub repository
REF = "main"  # branch or tag
REPO_DIR = REPO.split("/")[-1]
os.environ.update(REPO=REPO, REF=REF, REPO_DIR=REPO_DIR)

In [ ]:
# @title Install the project and download Spider
import codecs
import subprocess

script = """
set -eu
curl -fsSL https://raw.githubusercontent.com/$REPO/$REF/scripts/colab_setup.sh -o /tmp/colab_setup.sh
bash /tmp/colab_setup.sh
bash "/content/$REPO_DIR/scripts/download_data.sh" spider
"""
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
with subprocess.Popen(
    ["bash", "-c", script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT
) as process:
    while chunk := process.stdout.read1(1024):
        print(decoder.decode(chunk), end="", flush=True)
if process.returncode:
    raise RuntimeError(f"setup failed with exit code {process.returncode}")

In [ ]:
# @title Imports
%cd -q /content/{REPO_DIR}

import tqdm.auto as tqdm_auto
from tqdm import tqdm

tqdm_auto.tqdm = tqdm

import json
from datetime import UTC, datetime
from pathlib import Path

from text2sql.data import build_samples
from text2sql.meta import metadata_path, write_generation_metadata
from text2sql.model import MODELS, generate_batch, load_model
from text2sql.prompt import build_prompt

Samples of the split, and the model with its tokenizer loaded in 4-bit.

In [ ]:
# @title Load the samples and the model
if MODEL not in MODELS:
    raise ValueError(f"MODEL must be one of {MODELS}, not {MODEL!r}")
if SPLIT not in ("dev", "test"):
    raise ValueError(f"SPLIT must be dev or test, not {SPLIT!r}")

samples = build_samples(Path("data/spider_data"), SPLIT)
model, tokenizer = load_model(MODEL)

Predictions of every question in the split, generated in batches and appended to the predictions file. A rerun skips the questions already in it.

In [ ]:
# @title Generate the predictions
if BATCH_SIZE < 1:
    raise ValueError(f"BATCH_SIZE must be at least 1, not {BATCH_SIZE}")

started_at = datetime.now(UTC)
predictions_path = Path("predictions") / f"{MODEL.split('/')[-1].lower()}_{RUN}_{SPLIT}.jsonl"
predictions_path.parent.mkdir(exist_ok=True)
done = (
    {json.loads(line)["id"] for line in predictions_path.read_text().splitlines()}
    if predictions_path.exists()
    else set()
)

conversations = {i: build_prompt(sample) for i, sample in enumerate(samples) if i not in done}
todo = sorted(conversations, key=lambda i: sum(len(m["content"]) for m in conversations[i]))

with predictions_path.open("a") as file, tqdm(total=len(samples), initial=len(done)) as progress:
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start : start + BATCH_SIZE]
        replies = generate_batch(model, tokenizer, [conversations[i] for i in batch])
        for i, predicted in zip(batch, replies):
            file.write(
                json.dumps({"id": i, "db_id": samples[i]["db_id"], "predicted": predicted}) + "\n"
            )
        file.flush()
        progress.update(len(batch))

finished_at = datetime.now(UTC)
if todo:
    print(f"generated {len(todo)} predictions, {len(samples)} questions in {predictions_path}")
else:
    print(
        f"nothing was generated, all {len(samples)} predictions are already in {predictions_path}"
    )

Metadata of the run, saved next to the predictions file.

In [ ]:
# @title Write the run metadata
existing = metadata_path(predictions_path)
if len(done) == len(samples) and existing.exists():
    print(f"nothing was generated, keeping {existing}")
else:
    written = write_generation_metadata(
        predictions_path,
        model,
        run=RUN,
        split=SPLIT,
        batch_size=BATCH_SIZE,
        questions=len(samples),
        already_done=len(done),
        started_at=started_at,
        finished_at=finished_at,
    )
    print(f"metadata in {written}")